# NeXus data-collection scan catalogue

Build an in-memory pandas DataFrame from a directory of NeXus files. **Every input is opened with `h5py.File(path, 'r')`; no input files are changed, and this notebook has no export/write step.** Detector images and companion detector files are not read.

Only scans with `/entry/plan_metadata/plan_name == 'data_collection'` and a nonempty finite `/entry/fastcs_eiger/tth` array are included. Other plans and incompatible/unreadable files are listed in `skipped_files`.

Angle groups use the fast-correction notebook's minimum-anchored rule: sort unique motor angles and include an angle while `angle - group minimum <= ANGLE_TOLERANCE_DEG`. I0 is not used. Duration is calculated from start/end timestamps where available, otherwise from the recorded duration with its units. Timestamps retain the file's timezone convention (naive timestamps are not assumed to be UTC).

Sample metadata is expanded into `sample_*` columns, including name, composition, capillary, density, packing fraction, container and position when available. Missing metadata remains missing.

In [1]:
from pathlib import Path
import json
import re
from collections import Counter

import h5py
import numpy as np
import pandas as pd
from IPython.display import display

DATA_DIRECTORY = Path('/dls/i15-1/data/2026/cm44163-4')
RECURSIVE = False
ANGLE_TOLERANCE_DEG = 0.001
PLAN_NAME = 'data_collection'

if not DATA_DIRECTORY.is_dir():
    raise NotADirectoryError(DATA_DIRECTORY)
if not np.isfinite(ANGLE_TOLERANCE_DEG) or ANGLE_TOLERANCE_DEG < 0:
    raise ValueError('ANGLE_TOLERANCE_DEG must be finite and nonnegative')
print('Read-only input directory:', DATA_DIRECTORY)
print('Angle grouping tolerance (degrees):', ANGLE_TOLERANCE_DEG)

Read-only input directory: /dls/i15-1/data/2026/cm44163-4
Angle grouping tolerance (degrees): 0.001


In [2]:
def decode_value(value):
    if isinstance(value, bytes):
        return value.decode('utf-8', errors='replace')
    if isinstance(value, np.ndarray):
        if value.size == 1:
            return decode_value(value.reshape(-1)[0])
        return [decode_value(item) for item in value.tolist()]
    if isinstance(value, np.generic):
        return value.item()
    return value


def small_value(file, path, default=None):
    dataset = file.get(path)
    if not isinstance(dataset, h5py.Dataset) or dataset.size > 64:
        return default
    value = decode_value(dataset[()])
    return json.dumps(value, ensure_ascii=False) if isinstance(value, list) else value


def parse_timestamp(value):
    if value is None or not str(value).strip():
        return pd.NaT
    try:
        return pd.Timestamp(value)
    except (ValueError, TypeError, OverflowError):
        return pd.NaT


def scan_duration(file, start, end):
    if pd.notna(start) and pd.notna(end):
        try:
            elapsed = end - start
            if elapsed >= pd.Timedelta(0):
                return elapsed, 'end_time - start_time'
        except TypeError:
            pass  # Inconsistent timezone metadata: use the recorded duration.
    units = {
        's': 's', 'sec': 's', 'second': 's', 'seconds': 's',
        'ms': 'ms', 'millisecond': 'ms', 'milliseconds': 'ms',
        'us': 'us', 'microseconds': 'us', 'ns': 'ns',
        'min': 'min', 'minutes': 'min', 'h': 'h', 'hours': 'h',
    }
    for path in ('entry/duration', 'entry/diamond_scan/duration'):
        dataset = file.get(path)
        if not isinstance(dataset, h5py.Dataset) or dataset.size != 1:
            continue
        unit = units.get(str(decode_value(dataset.attrs.get('units', ''))).lower())
        if unit is None:
            continue  # Never guess units for a recorded numeric duration.
        try:
            value = float(np.asarray(dataset[()]).reshape(-1)[0])
            if np.isfinite(value) and value >= 0:
                return pd.to_timedelta(value, unit=unit), path
        except (ValueError, TypeError, OverflowError):
            continue
    return pd.NaT, 'unavailable'


def sample_columns(file):
    columns = {}
    # Inspect only sample metadata; do not walk detector groups/external links.
    for root, prefix in (('entry/plan_metadata/sample_info', 'sample'),
                         ('entry/sample', 'nexus_sample')):
        group = file.get(root)
        if not isinstance(group, h5py.Group):
            continue
        def collect(name, obj):
            if isinstance(obj, h5py.Dataset) and obj.size <= 64:
                key = prefix + '_' + name.replace('/', '_')
                value = decode_value(obj[()])
                columns[key] = json.dumps(value, ensure_ascii=False) if isinstance(value, list) else value
        group.visititems(collect)
    return columns


def count_angle_groups(angles, tolerance):
    unique_angles = np.unique(angles)
    count = 0
    anchor = None
    for angle in unique_angles:
        if anchor is None or angle - anchor > tolerance:
            anchor = angle
            count += 1
    return count


def summarise_scan(path):
    with h5py.File(path, 'r') as file:
        if small_value(file, 'entry/plan_metadata/plan_name') != PLAN_NAME:
            return None, 'not the requested plan'
        dataset = file.get('entry/fastcs_eiger/tth')
        if not isinstance(dataset, h5py.Dataset):
            return None, 'missing EIGER tth array'
        angles = np.asarray(dataset[()], dtype=float).reshape(-1)
        if not angles.size or not np.all(np.isfinite(angles)):
            return None, 'empty or nonfinite tth array'
        start = parse_timestamp(small_value(file, 'entry/start_time',
                                small_value(file, 'entry/diamond_scan/start_time')))
        end = parse_timestamp(small_value(file, 'entry/end_time',
                              small_value(file, 'entry/diamond_scan/end_time')))
        duration, duration_source = scan_duration(file, start, end)
        match = re.search(r'([0-9]+)$', path.stem)
        identifier = small_value(file, 'entry/diamond_scan/entry_identifier')
        scan_number = int(match.group(1)) if match else (
            int(identifier) if identifier is not None and str(identifier).isdigit() else pd.NA)
        row = {
            'scan_number': scan_number, 'start_time': start, 'end_time': end,
            'duration': duration,
            'duration_seconds': duration.total_seconds() if pd.notna(duration) else np.nan,
            'tth_grouped_angles': count_angle_groups(angles, ANGLE_TOLERANCE_DEG),
            'max_tth_deg': float(angles.max()),
            'sample_name': None, 'sample_data_composition': None,
            'sample_data_capillary': None, 'sample_data_density': None,
            'sample_data_packing_fraction': None,
            'raw_frames': len(angles), 'exact_tth_angles': len(np.unique(angles)),
            'angle_tolerance_deg': ANGLE_TOLERANCE_DEG,
            'duration_source': duration_source, 'filename': path.name, 'path': str(path),
        }
        row.update(sample_columns(file))
        return row, None

In [ ]:
def format_duration(seconds):
    """Display total minutes and seconds, rounded to the nearest second."""
    if pd.isna(seconds):
        return pd.NA
    minutes, seconds = divmod(int(np.floor(float(seconds) + 0.5)), 60)
    return f'{minutes} min {seconds:02d} s'


assert format_duration(3772.406) == '62 min 52 s'
assert format_duration(59.9) == '1 min 00 s'
assert format_duration(0) == '0 min 00 s'
assert pd.isna(format_duration(np.nan))

nexus_paths = sorted(DATA_DIRECTORY.rglob('*.nxs') if RECURSIVE else DATA_DIRECTORY.glob('*.nxs'))
rows, skipped = [], []
for path in nexus_paths:
    try:
        row, reason = summarise_scan(path)
        if row is not None:
            rows.append(row)
        else:
            skipped.append({'filename': path.name, 'path': str(path), 'reason': reason})
    except (OSError, KeyError, ValueError, TypeError, RuntimeError, OverflowError) as error:
        skipped.append({'filename': path.name, 'path': str(path),
                        'reason': f'{type(error).__name__}: {error}'})

leading_columns = [
    'scan_number', 'start_time', 'end_time', 'duration', 'duration_seconds',
    'tth_grouped_angles', 'max_tth_deg', 'sample_name',
    'sample_data_composition', 'sample_data_capillary', 'sample_data_density',
    'sample_data_packing_fraction',
]
scan_summary = pd.DataFrame(rows)
for column in leading_columns:
    if column not in scan_summary:
        scan_summary[column] = pd.Series(dtype='object')
scan_summary['duration'] = scan_summary['duration_seconds'].map(format_duration)
scan_summary['scan_number'] = pd.array(scan_summary['scan_number'], dtype='Int64')
if scan_summary['scan_number'].isna().any():
    raise ValueError('Cannot use scan_number as the index: some matching files have no scan number.')
if not scan_summary['scan_number'].is_unique:
    duplicate_numbers = scan_summary.loc[scan_summary['scan_number'].duplicated(keep=False), 'scan_number'].tolist()
    raise ValueError(f'Cannot use scan_number as a primary key: duplicate scan numbers {duplicate_numbers}')
remaining_columns = [column for column in scan_summary if column not in leading_columns]
# Treat scan_number as a primary key; never silently discard duplicate scans.
scan_summary = scan_summary[leading_columns + remaining_columns].set_index('scan_number').sort_index()
skipped_files = pd.DataFrame(skipped, columns=['filename', 'path', 'reason'])
print(f'Inspected {len(nexus_paths):,} files: included {len(scan_summary):,}; skipped {len(skipped_files):,}.')
print('Skipped-file reasons:', dict(Counter(skipped_files['reason'])))
with pd.option_context('display.max_columns', None, 'display.max_rows', 100):
    display(scan_summary)

## Inspect or filter the results

`scan_summary` contains all matching scans, indexed by the unique **scan number** (the display may truncate a long table). Select a scan with `scan_summary.loc[100603]`, or a scan-number range with `scan_summary.loc[100599:100609]`. Duplicate or missing scan numbers raise an error rather than creating an ambiguous index.

`skipped_files` records every excluded file and its reason. Rerun the catalogue cell to refresh after more scans finish. Files still being acquired can have missing end times; those scans remain in the table.

Use ordinary pandas filtering, for example `scan_summary.loc[scan_summary['sample_name'].str.contains('silicon', case=False, na=False)]`. The following compact view keeps sample descriptions beside the main scan fields. No files are written.

In [ ]:
# Basic consistency checks, including minimum-anchored rather than chained grouping.
assert count_angle_groups(np.array([0., 0., 0.5, 1.]), 0.75) == 2
assert count_angle_groups(np.array([0., 0., 0.5, 1.]), 0.) == 3
assert count_angle_groups(np.array([0., 0.5, 1.]), 0.5) == 2
assert count_angle_groups(np.array([]), 0.001) == 0
assert len(scan_summary) + len(skipped_files) == len(nexus_paths)
assert scan_summary.index.name == 'scan_number'
assert scan_summary.index.is_unique
assert not scan_summary.index.hasnans
if not scan_summary.empty:
    assert (scan_summary['tth_grouped_angles'] <= scan_summary['exact_tth_angles']).all()
    assert (scan_summary['exact_tth_angles'] <= scan_summary['raw_frames']).all()
    assert (scan_summary['duration_seconds'].dropna() >= 0).all()

# Compact view; scan_number is the index, not a duplicate data column.
display(scan_summary[[
    'start_time', 'end_time', 'duration',
    'tth_grouped_angles', 'max_tth_deg', 'sample_name',
    'sample_data_composition', 'sample_data_capillary',
]])

,scan_number,start_time,end_time,duration,tth_grouped_angles,max_tth_deg,sample_name,sample_data_composition,sample_data_capillary
0,98454,2026-09-04 12:33:58.463,2026-09-04 12:34:24.117,0 min 26 s,6,59.999995,Empty bs1.0,NaN,bs1.0
1,98458,2026-09-04 13:25:11.342,2026-09-04 13:27:17.797,2 min 06 s,6,60.000051,Empty fq3.17,NaN,fq3.17
2,98472,2026-09-08 14:27:30.190,2026-09-08 14:27:56.976,0 min 27 s,6,59.999995,Empty bs1.0,NaN,bs1.0
3,98476,2026-09-08 15:08:51.940,2026-09-08 15:09:20.572,0 min 29 s,6,59.999995,Empty bs1.0,NaN,bs1.0
4,98478,2026-09-08 15:11:26.052,2026-09-08 15:11:54.580,0 min 29 s,6,59.999995,Alice,SiO2,bs1.0
...,...,...,...,...,...,...,...,...,...
163,100601,2026-10-16 01:39:36.646,2026-10-16 02:42:30.531,62 min 54 s,41,79.999975,Silicon bs1.0,Silicon,bs1.0
164,100603,2026-10-16 02:44:50.668,2026-10-16 03:47:43.074,62 min 52 s,41,79.999975,Playdate - ground,K2PdCl6,bs1.0
165,100605,2026-10-16 03:50:05.803,2026-10-16 04:52:59.731,62 min 54 s,41,79.999975,C60,C,bs1.0
166,100607,2026-10-16 04:55:19.649,2026-10-16 05:58:12.338,62 min 53 s,41,80.000030,Glass beads,SiO2,bs1.0
